This is single document summarization application using MapReduce Architecture.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !pip install langchain transformers sentence-transformers
!pip install langchain transformers accelerate bitsandbytes pypdf sentence-transformers faiss-cpu PyPDF2 tiktoken
!pip install langchain-community
!pip install langchain-huggingface
!pip install -U transformers
!pip install -U transformers langchain sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requ

In [3]:
## Document Loader and Chunking
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters  import RecursiveCharacterTextSplitter
from langchain_core.documents  import Document
from sentence_transformers import SentenceTransformer
import PyPDF2


# ========================
# 1. Read PDF
# ========================
def read_pdf(file_path):
    pdf_reader = PyPDF2.PdfReader(file_path)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text() + "\n"
    return text

pdf_text = read_pdf("/content/drive/MyDrive/LangChain Learning/document/responsible_finance.pdf")

# ========================
# 2. Semantic Chunking
# ========================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,   #conventional size is used
    chunk_overlap=50,
    length_function=len
)
chunks = text_splitter.split_text(pdf_text)

# ========================
# 3. Store chunks as Documents with metadata
# ========================
documents = []
for i, chunk in enumerate(chunks):
    doc = Document(
        page_content=chunk,
        metadata={
            "chunk_id": i,
            "source": "responsible_finance.pdf"
        }
    )
    documents.append(doc)

print(f"Total chunks created: {len(documents)}")
print(documents[0].metadata, documents[0].page_content[:100], "...")

document_sections = []
for document in documents:
    document_sections.append(Document(page_content=document.page_content))
print('Number of Chunks: ', len(document_sections))
document_sections[0], type(document_sections[0])

Total chunks created: 152
{'chunk_id': 0, 'source': 'responsible_finance.pdf'} Early microfinance pioneers created the sector 
with the aspiration to improve poor people’s 
lives. ...
Number of Chunks:  152


(Document(metadata={}, page_content='Early microfinance pioneers created the sector \nwith the aspiration to improve poor people’s \nlives. There were huge swathes of “white space” where low-income households had little or no \naccess to formal finance. This challenge, in turn, \ndemanded sustainable and scalable models for \ndelivering adequate financial services to poor \npeople. \nThis is the essence of the double bottom line in \nmicrofinance: a social commitment to benefiting \nclients married with a financial commitment to'),
 langchain_core.documents.base.Document)

In [4]:
# Imports
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain, MapReduceChain, load_summarize_chain
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from typing import List

# ========================
# 1. Setup LLM (HuggingFace free model)
# ========================
hf_pipeline = pipeline(
    task="text-generation",
    model="google/flan-t5-base",  # Free model, smaller & fast
    max_new_tokens=256,
    temperature = 0.1,
    max_length=None
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'Deepse

In [5]:
# ========================
# 2. Define Map and Reduce Prompts
# ========================
map_prompt = PromptTemplate(
    input_variables=["text"],
    template=""" Summarize the following text in 3 concise sentences.\n\n {text}"""
)

reduce_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Combine the following summaries into a clear and concise final summary:\n\n{text}"""
)

# ========================
# 3. Define Map and Reduce Chains
# ========================
map_chain = LLMChain(llm=llm, prompt=map_prompt)
reduce_chain = LLMChain(llm=llm, prompt=reduce_prompt)

# ========================
# 4. MapReduceChain
# ========================
summary_chain = load_summarize_chain(
    llm,
    chain_type="map_reduce", ## As MapReduce Architecture Used
    map_prompt=map_prompt,
    combine_prompt=reduce_prompt
)

# ========================
# 5. Run MapReduce Summarization
# ========================
summary_result = summary_chain.invoke(document_sections[5:10]) ## Passing five paragrams to save resources
print("=== FINAL SUMMARY ===")
print(summary_result)

/tmp/ipykernel_904/3711912946.py:17: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  map_chain = LLMChain(llm=llm, prompt=map_prompt)
/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

=== FINAL SUMMARY ===
{'input_documents': [Document(metadata={}, page_content='delivery. We need to better manage the trade-\noffs to ensure that products and practices are \nsound and client welfare and institutional viability \nare not put at risk. Scale and sustainability goals \nremain legitimate and, indeed, critical goals, but they must be supported with the necessary \ninfrastructure, such as credit bureaus, and \npublic goods, such as transparency regulation.\n2\nIn sum, we need to rebalance the double bottom line as our collective goal shifts from simply filling'), Document(metadata={}, page_content='in the white space to developing a healthy and \nresponsible market. \nAgainst this backdrop of high aspiration and new \nchallenges, the microfinance field has embraced \nthe need for a proactive responsible finance agenda \nand accelerated its efforts to implement meaningful \nimprovements in products and practices. There is a growing consensus that providers of financial \nserv

In [6]:
print(summary_result['output_text'])

Combine the following summaries into a clear and concise final summary:

 Summarize the following text in 3 concise sentences.

 delivery. We need to better manage the trade-
offs to ensure that products and practices are 
sound and client welfare and institutional viability 
are not put at risk. Scale and sustainability goals 
remain legitimate and, indeed, critical goals, but they must be supported with the necessary 
infrastructure, such as credit bureaus, and 
public goods, such as transparency regulation.
2
In sum, we need to rebalance the double bottom line as our collective goal shifts from simply filling

 Summarize the following text in 3 concise sentences.

 in the white space to developing a healthy and 
responsible market. 
Against this backdrop of high aspiration and new 
challenges, the microfinance field has embraced 
the need for a proactive responsible finance agenda 
and accelerated its efforts to implement meaningful 
improvements in products and practices. There is 